# 1. Data Exploration
Explore the bias datasets and visualize statistics about the prompt pairs.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Load Datasets

In [ ]:
with open('../data/gender_bias.json', 'r') as f:
    gender_data = json.load(f)

with open('../data/demographic_bias.json', 'r') as f:
    demo_data = json.load(f)

print(f'Gender bias prompts: {len(gender_data)}')
print(f'Demographic bias prompts: {len(demo_data)}')

## Gender Bias: Clean vs Corrupted Pairs

In [ ]:
gender_df = pd.DataFrame(gender_data)
gender_df[['biased_word', 'neutral_word', 'expected_bias_direction', 'clean_prompt']].head(10)

In [ ]:
# Visualize bias direction distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender bias direction
direction_counts = gender_df['expected_bias_direction'].value_counts()
colors = ['#4A90D9', '#E74C8B']
axes[0].pie(direction_counts.values, labels=direction_counts.index, autopct='%1.0f%%',
            colors=colors, startangle=90, textprops={'fontsize': 14})
axes[0].set_title('Gender Bias Direction Distribution', fontsize=14, fontweight='bold')

# Prompt length distribution
gender_df['clean_len'] = gender_df['clean_prompt'].str.split().str.len()
gender_df['corrupted_len'] = gender_df['corrupted_prompt'].str.split().str.len()
axes[1].hist([gender_df['clean_len'], gender_df['corrupted_len']], bins=8,
             label=['Clean', 'Corrupted'], color=['#5B8FB9', '#B6533A'], alpha=0.8)
axes[1].set_xlabel('Number of words')
axes[1].set_ylabel('Count')
axes[1].set_title('Prompt Length Distribution', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/data_exploration_gender.png', dpi=150, bbox_inches='tight')
plt.show()

## Demographic Bias: Prompt Pairs

In [ ]:
demo_df = pd.DataFrame(demo_data)
demo_df[['biased_word', 'neutral_word', 'clean_prompt']].head(10)

In [ ]:
# Word cloud of biased words
fig, ax = plt.subplots(figsize=(10, 4))

all_biased_words = list(gender_df['biased_word']) + list(demo_df['biased_word'])
all_neutral_words = list(gender_df['neutral_word']) + list(demo_df['neutral_word'])

y_pos = range(len(all_biased_words))
ax.barh(y_pos, [1]*len(all_biased_words), color='#E74C3C', alpha=0.7, label='Biased')
for i, word in enumerate(all_biased_words):
    ax.text(0.5, i, f'{word} → {all_neutral_words[i]}', ha='center', va='center', fontsize=8)

ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Biased → Neutral Word Substitutions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/data_exploration_substitutions.png', dpi=150, bbox_inches='tight')
plt.show()

## Load and Visualize Baseline Scores (if available)

In [ ]:
import os

baseline_path = '../results/baseline_gender_bias.json'
if os.path.exists(baseline_path):
    with open(baseline_path, 'r') as f:
        baseline = json.load(f)
    
    scores_df = pd.DataFrame(baseline['per_prompt'])
    
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#4A90D9' if d == 'male' else '#E74C8B' for d in scores_df['direction']]
    bars = ax.bar(range(len(scores_df)), scores_df['bias_score'], color=colors, alpha=0.85)
    ax.axhline(y=baseline['mean_bias'], color='red', linestyle='--', label=f'Mean: {baseline["mean_bias"]:.3f}')
    ax.set_xlabel('Prompt Index')
    ax.set_ylabel('L2 Bias Score')
    ax.set_title('Baseline Bias Scores per Prompt', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/baseline_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'No baseline results found at {baseline_path}.')
    print('Run: python scripts/01_run_baseline.py --dataset data/gender_bias.json')